<a href="https://colab.research.google.com/github/elianafonseca/GEO01007/blob/main/SIG_tabelas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # Análise Espaço-temporal da Produção de Leite no Rio Grande do Sul

Este notebook tem como objetivo realizar uma análise espaço-temporal da produção de leite nos municípios do Rio Grande do Sul, comparando os dados dos anos de 2001 e 2011. Para isso, utilizaremos dados geográficos (shapefile) e dados tabulares de produção de leite, visualizando-os em mapas interativos e estáticos.

## 1. Acesso ao Google Drive

Primeiramente, precisamos montar o Google Drive para acessar os arquivos necessários (shapefile e CSV) que estão armazenados lá.

In [ ]:
# acesso ao Google drive
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

## 2. Instalação e Importação de Bibliotecas

Para trabalhar com dados geoespaciais no Python, utilizaremos a biblioteca `geopandas`, que estende as funcionalidades do `pandas` para dados geográficos.

Depois de instalar, importamos a biblioteca `geopandas`.

Precisamos também instalar a biblioteca `geojson`, que é útil para trabalhar com o formato de dados GeoJSON, muitas vezes usado em visualizações de mapas interativos.

In [ ]:
# Instalar geopandas
!pip install geopandas -q

import geopandas as gpd

In [ ]:
# Instalar geojson
!pip install geojson -q

## 3. Carregamento dos Dados Geográficos (Shapefile)


Vamos carregar o shapefile que contém a geometria dos municípios do Rio Grande do Sul. Usamos `gpd.read_file()` para ler o arquivo e o comando `data.head()` para inspecionar as primeiras cinco linhas e entender a estrutura dos dados.

In [ ]:
# Ler shapefile
data = gpd.read_file('/content/drive/MyDrive/shapes/43mu2500gsd.shp')
# Inspecionar os dados
data.head()

## 3. Carregamento da Tabela
Em seguida, carregamos o arquivo CSV que contém os dados de produção de leite (`leite(mil_litros).csv`). Utilizamos `pd.read_csv()` e especificamos a codificação (`latin1`) e o separador (`;`) para garantir a leitura correta. `leite_df.head()` mostra as primeiras linhas.

In [ ]:
import pandas as pd

file_path = '//content/drive/MyDrive/shapes/leite(mil_litros).csv'
leite_df = pd.read_csv(file_path, encoding='latin1', sep=';')
leite_df.head()

## 4. Junção dos Dados

Agora, vamos combinar os dados geográficos (`data`) com os dados de produção de leite (`leite_df`). Usamos a função `merge()` do pandas, especificando as colunas de junção (`ordem_alfa` em `data` e `alf` em `leite_df`) e o tipo de junção (`how='left'`). A coluna `alf` parece ser um identificador único para cada município na tabela de leite.

In [ ]:
# Merge (juntar) o geodataframe com o dataframe usando as colunas alf e ordem_alfa
merged = data.merge(leite_df, left_on='ordem_alfa', right_on='alf', how='left')

# Inspecionar o resultado
display(merged.head())

Selecionamos as colunas de interesse para a análise final: o `GEOCODIG_M` (o identificador único para o mapa), `ano_2001`, `ano_2011` (dados de produção de leite) e a `geometry` (a informação espacial de cada município). Isso cria um GeoDataFrame mais limpo para as visualizações.

In [ ]:
# Selecionar apenas as colunas de interesse
data_final = merged[[ 'GEOCODIG_M', 'Nome_Munic','ano_2001', 'ano_2011', 'geometry']]

# Verifique o resultado
data_final.head()

## 5. Visualização com Mapas Estáticos (Matplotlib/GeoPandas)

Para criar mapas estáticos com legendas mais customizáveis e esquemas de classificação, utilizaremos a biblioteca `mapclassify` em conjunto com `matplotlib` e `geopandas`. Primeiro, instalamos `mapclassify`.

In [ ]:
# Instalar a biblioteca mapclassify
!pip install mapclassify==2.10.0 -q

Importamos a biblioteca `matplotlib.pyplot`, que é a principal biblioteca para criar gráficos e visualizações estáticas em Python.

In [ ]:
import matplotlib.pyplot as plt

Criamos um mapa coroplético da produção de leite nos dois anos usando `data_final.plot()`. Definimos `bins` (intervalos personalizados) para classificar a produção e usamos o esquema `UserDefined` com o mapa de cores `Reds` para visualizar as áreas de maior e menor produção. A legenda ajuda a interpretar as cores.

In [ ]:
# criar figura e eixos de plotagem
fig, ax = plt.subplots(1, figsize=(10, 8))

# Definir os intervalos fixos (bins) que você deseja usar
bins = [ 1000, 5000, 50000]

# Filter out NaN values for the column being plotted
data_final_filtered = data_final.dropna(subset=['ano_2001'])

# Crie um novo plot usando classificação (scheme) por intervalos fixos
data_final_filtered.plot(ax = ax, column="ano_2001", scheme="UserDefined", classification_kwds={'bins': bins}, cmap="Reds", legend=True, legend_kwds={'loc': 'lower left'}, edgecolor='black')

# Adicione um título ao mapa (opcional)
ax.set_title('Produção de Leite em 2001 (mil litros)')

# Remova os eixos para um visual mais limpo (opcional)
#ax.set_axis_off()

In [ ]:
# criar figura e eixos de plotagem
fig, ax = plt.subplots(1, figsize=(10, 8))

# Definir os intervalos fixos (bins) que você deseja usar
bins = [ 1000, 5000, 50000]

# Filter out NaN values for the column being plotted
data_final_filtered = data_final.dropna(subset=['ano_2011'])

# Crie um novo plot usando classificação (scheme) por intervalos fixos
data_final_filtered.plot(ax = ax, column="ano_2011", scheme="UserDefined", classification_kwds={'bins': bins}, cmap="Reds", legend=True, legend_kwds={'loc': 'lower left'}, edgecolor='black')

# Adicione um título ao mapa (opcional)
ax.set_title('Produção de Leite em 2011 (mil litros)')

# Remova os eixos para um visual mais limpo (opcional)
#ax.set_axis_off()

## 6. Análise da Variação da Produção de Leite

Para entender as mudanças na produção de leite entre 2001 e 2011, tem duas abordagens. Calcular a diferença entre o valor de um ano e de outro OU calcular a mudança de classes (bins).

### 6.1. Análise por Diferença Absoluta de Produção

Nesta abordagem, calculamos a diferença direta entre a produção de leite em 2011 e 2001. Uma nova coluna, `diferenca_producao`, é criada para armazenar essa variação. Em seguida, categorizamos essa diferença em `Aumento`, `Diminuição` ou `Estável` para facilitar a análise.


In [ ]:
# Calcular a diferença na produção de leite entre 2011 e 2001
data_final['diferenca_producao'] = data_final['ano_2011'] - data_final['ano_2001']

# Criar uma nova coluna indicando aumento, diminuição ou estabilidade
data_final['status_producao'] = 'Estável' # Valor padrão

# Usar numpy.select para categorizar a diferença
import numpy as np
conditions = [
    (data_final['diferenca_producao'] > 0),
    (data_final['diferenca_producao'] < 0)
]
choices = ['Aumento', 'Diminuição']

data_final['status_producao'] = np.select(conditions, choices, default='Estável')

# Visualizar as novas colunas e algumas linhas do dataframe mesclado
display(data_final[['GEOCODIG_M', 'ano_2001', 'ano_2011', 'diferenca_producao', 'status_producao']].head())

### 6.2. Visualização da Diferença Absoluta

Para visualizar o `status_producao`, criamos um mapa estático onde municípios com aumento de produção são representados em verde, diminuição em vermelho, e aqueles com dados insuficientes ou estáveis em branco (ou cinza). Uma legenda manual é adicionada para facilitar a interpretação.

In [ ]:
# Crie uma figura e eixos de plotagem
fig, ax = plt.subplots(1, figsize=(12, 10))

# Defina um dicionário de cores para os status
color_dict = {'Aumento': 'green', 'Diminuição': 'red', 'Estável': 'white'}

# Mapeie as cores para a coluna 'status_producao'
data_final['color'] = data_final['status_producao'].map(color_dict)

# Crie o plot usando a nova coluna de cores
data_final.plot(ax=ax, color=data_final['color'], legend=True, edgecolor='black')

# Adicione um título ao mapa
ax.set_title('Status da Produção de Leite (2001-2011)')

# Remova os eixos
ax.set_axis_off()

# Crie uma legenda manual, pois plotar por cor não gera legenda automática para categorias
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=color_dict['Aumento'], label='Aumento'),
                   Patch(facecolor=color_dict['Diminuição'], label='Diminuição'),
                   Patch(facecolor=color_dict['Estável'], label='Sem dados')]
ax.legend(handles=legend_elements, loc='lower left')

plt.show()

### 6.3. Análise por Mudança de Classes (Bins)

Alternativamente, podemos categorizar a produção de leite em classes (bins) para cada ano e analisar a mudança entre essas classes. A função `categorize_production_by_bins` classifica a produção em diferentes níveis (por exemplo, baixa, média, alta). Em seguida, comparamos as classes de 2001 e 2011 para determinar se houve `Aumento de Classe`, `Diminuição de Classe`, `Sem Mudança de Classe` ou `Dados Incompletos`.

In [ ]:
import numpy as np

def categorize_production_by_bins(value, bins):
    """
    Categoriza um valor de produção com base nos bins fornecidos.
    Retorna um número inteiro representando a classe ou NaN se o valor for nulo.
    """
    if pd.isna(value):
        return np.nan
    # Adjusted to ensure 0 falls into the first bin if not handled by a specific 0 bin
    # For positive values, the logic holds. Assuming bins are [low, mid, high]
    if value <= bins[0]: # 1000
        return 1
    elif value <= bins[1]: # 5000
        return 2
    elif value <= bins[2]: # 50000
        return 3
    else:
        return 4

# Usar os mesmos bins definidos para os mapas estáticos
# Note: if 0 is an expected lower bound, a bin like [0, 1000,...] might be more explicit
current_bins = [1000, 5000, 50000]

# Aplicar a categorização para ambos os anos
data_final.loc[:, 'classe_2001'] = data_final['ano_2001'].apply(lambda x: categorize_production_by_bins(x, current_bins))
data_final.loc[:, 'classe_2011'] = data_final['ano_2011'].apply(lambda x: categorize_production_by_bins(x, current_bins))

# Criar uma nova coluna para indicar a mudança de classe
# 'Sem Mudança' se as classes forem iguais e não-NaN
# 'Aumento de Classe' se classe_2011 > classe_2001
# 'Diminuição de Classe' se classe_2011 < classe_2001
# 'Dados Incompletos' se alguma das classes for NaN

conditions = [
    (data_final['classe_2011'] > data_final['classe_2001']),
    (data_final['classe_2011'] < data_final['classe_2001']),
    (data_final['classe_2001'].isna() | data_final['classe_2011'].isna())
]
choices = ['Aumento de Classe', 'Diminuição de Classe', 'Dados Incompletos']

data_final.loc[:, 'mudanca_classe'] = np.select(conditions, choices, default='Sem Mudança de Classe')

### 6.4. Visualização da Mudança de Classes

Este mapa visualiza a `mudanca_classe`. Municípios com `Aumento de Classe` são exibidos em verde, `Diminuição de Classe` em vermelho, `Sem Mudança de Classe` em cinza e `Dados Incompletos` em branco. Esta visualização ajuda a identificar regiões que melhoraram ou pioraram sua categoria de produção ao longo do tempo.

In [ ]:
# Crie uma figura e eixos de plotagem
fig, ax = plt.subplots(1, figsize=(12, 10))

# Defina um dicionário de cores para os status da mudança de classe
color_dict_class_change = {
    'Aumento de Classe': 'green',
    'Diminuição de Classe': 'red',
    'Sem Mudança de Classe': 'gray',
    'Dados Incompletos': 'white' # Cor para municípios com dados incompletos
}

# Mapeie as cores para a coluna 'mudanca_classe'
data_final.loc[:, 'color_mudanca_classe'] = data_final['mudanca_classe'].map(color_dict_class_change)

# Crie o plot usando a nova coluna de cores
data_final.plot(ax=ax, color=data_final['color_mudanca_classe'], edgecolor='black', linewidth=0.5)

# Adicione um título ao mapa
ax.set_title('Status de Mudança da Classe de Produção de Leite (2001-2011)')

# Remova os eixos
ax.set_axis_off()

# Crie uma legenda manual
from matplotlib.patches import Patch
legend_elements_class_change = [
    Patch(facecolor=color_dict_class_change['Aumento de Classe'], label='Aumento de Classe'),
    Patch(facecolor=color_dict_class_change['Diminuição de Classe'], label='Diminuição de Classe'),
    Patch(facecolor=color_dict_class_change['Sem Mudança de Classe'], label='Sem Mudança de Classe'),
    Patch(facecolor=color_dict_class_change['Dados Incompletos'], label='Dados Incompletos')
]
ax.legend(handles=legend_elements_class_change, loc='lower left', title='Legenda')

plt.show()

## 7. Visualização com Mapas Interativos (Folium)

Importamos a biblioteca `folium`, que permite criar mapas interativos que podem ser facilmente incorporados em notebooks ou páginas web. É uma ótima ferramenta para explorar dados geográficos de forma dinâmica.

In [ ]:
import folium

### 7.1. Geração do Geo-ID para Folium

Para facilitar a integração com a biblioteca Folium para mapas interativos, criamos uma nova coluna `geo_id` a partir do índice do GeoDataFrame. Esta coluna servirá como um identificador geográfico (geo-id) que o Folium pode usar para vincular as feições geográficas aos dados tabulares.

In [ ]:
# Criar um "geo-id" necessário  para a  biblioteca folium
data_final['geo_id'] = data.index.astype(str)

### 7.2. Reprojeção do Sistema de Coordenadas

Verificar o Sistema de Referência de Coordenadas (CRS) do GeoDataFrame. O comando `merged.crs` mostra qual CRS está sendo usado atualmente. Para padronização e compatibilidade com ferramentas de mapeamento web como o Folium, muitas vezes é necessário reprojetar para um CRS global comum, como o EPSG:4326 (WGS 84), que usa coordenadas de latitude e longitude.

In [ ]:
# Visualizar as novas colunas e algumas linhas do dataframe
display(data_final[['geo_id', 'Nome_Munic', 'ano_2001', 'classe_2001', 'ano_2011', 'classe_2011', 'mudanca_classe']].head())

In [ ]:
# verifique o sistema de coordenadas
merged.crs

Aqui, realizamos a reprojeção do GeoDataFrame `merged` para o sistema de coordenadas EPSG:4326. Isso garante que os dados estejam no formato correto para visualização em mapas interativos.

In [ ]:
# Reprojetar a camada
merged=merged.to_crs('epsg:4326')

In [ ]:
merged


In [ ]:
# Criar mapa
m = folium.Map(location=[-30, -51],  zoom_start=7, control_scale=True)

# adicionar o mapa coroplético ao mapa base
# A coluna geoid PRECISA ser sempre a primeira ser indicada nas columns
folium.Choropleth(
    geo_data=data_final,
    name='Produção leite 2001',
    data=data_final,
    columns=['geo_id', 'ano_2011'],
    key_on='feature.id',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    line_color='black',
    line_weight=0.5,
    highlight=False,
    smooth_factor=1.0,
    legend_name= 'Leite (mil litros)').add_to(m)

# Visualize o resultado
m

### 7.3. Criação do Mapa Coroplético

Aqui, criamos um mapa base usando `folium.Map()`, centralizando-o em uma coordenada específica do Rio Grande do Sul e definindo um nível de zoom. Em seguida, adicionamos uma camada coroplética (`folium.Choropleth`) que visualiza a produção de leite em 2011. A cor das áreas dos municípios varia de acordo com os valores da coluna `ano_2011`.

### 7.4. Adição de Tooltips Interativos

**Tooltips**

É possível adicionar diferentes tipos de mensagens pop-up e dicas de ferramentas ao mapa. Aqui, seria bom ver a área de cada distrito quando você passa o mouse sobre o mapa. Infelizmente, esta funcionalidade aparentemente não está implementada no método Choropleth que usamos antes.

Para uma solução rápida, plotamos os polígonos no topo do mapa como uma camada transparente e adicionamos a dica de ferramenta a esses objetos.

Esta não é uma solução ideal, pois agora a geometria do polígono é armazenada duas vezes na saída!

In [ ]:
# Aqui convertemos nosso pontos para geojson e os adicionamso como uma camada transparente. Assim teremos apenas o pop-up mostrando as infromações passadas ao tooltips
folium.features.GeoJson(data_final,
                        name='Labels',
                        style_function= lambda x:{'color':'transparent', 'fillcolor':'transparent',
                                                  'weight':0},
                        tooltip= folium.features.GeoJsonTooltip(fields=['ano_2001'],
                                                                aliases=['Produção (mil litros)'],
                                                                labels=True,
                                                                sticky=False)).add_to(m)
# Visualize o resultado
m

Para adicionar informações interativas ao mapa, como tooltips (pop-ups com dados ao passar o mouse), usamos `folium.features.GeoJson`. Plotamos os dados como uma camada transparente por cima do mapa coroplético e configuramos `GeoJsonTooltip` para exibir a produção de 2001 quando o usuário interage com o mapa.